# 5 · Limiar de detecção e avaliação final

O limiar de confiança é escolhido **somente na validação**, pela melhor pontuação F1 de caixas em IoU ≥ 0,5; a contagem exata entre imagens positivas desempata. Depois é aplicado, sem ajustes, ao teste. Ambas as partições são as do notebook 1. Imagens sem resistor entram em precisão, contagem exata e erro absoluto; a contagem exata também é mostrada separadamente nas imagens positivas.

In [ ]:
from pathlib import Path
import os,json,time
import cv2,numpy as np,pandas as pd,torch
from ultralytics import YOLO
def read_image(path):
    try: return cv2.imdecode(np.fromfile(str(path),dtype=np.uint8),cv2.IMREAD_COLOR)
    except (OSError,cv2.error): return None
OUTPUT=Path(os.environ.get("RESISTOR_OUTPUT","../outputs/proposta")).expanduser().resolve()
DATASET=OUTPUT/"samu-agrupado"; manifest=pd.read_csv(DATASET/"manifesto.csv")
sample_limit=int(os.environ.get("RESISTOR_METRICS_SAMPLE","0"))
if sample_limit:
    manifest=pd.concat([g.sample(n=min(sample_limit,len(g)),random_state=42) for _,g in manifest.groupby("split")])
    print("ATENÇÃO: amostra reduzida para teste técnico, não usar como resultado final.")
weights=Path(os.environ.get("RESISTOR_YOLO_WEIGHTS",str(OUTPUT/"treino/yolo-samu/weights/best.pt")))
assert weights.exists(), "Execute o treino do notebook 1."
model=YOLO(str(weights)); device=0 if torch.cuda.is_available() else "cpu"
print("Pesos:",weights,"dispositivo:",device)


In [ ]:
def iou(a,b):
    ix=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    return ix/max(1e-9,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-ix)
def match(gt,pred):
    pairs=sorted(((iou(a,b),i,j) for i,a in enumerate(gt) for j,b in enumerate(pred)),reverse=True)
    used_a,used_b=set(),set()
    for overlap,i,j in pairs:
        if overlap<.5: break
        if i not in used_a and j not in used_b:used_a.add(i);used_b.add(j)
    return len(used_a),len(pred)-len(used_b),len(gt)-len(used_a)
def infer_split(split):
    rows=[]
    for row in manifest[manifest.split==split].itertuples(index=False):
        image_path=next((DATASET/split/"images").glob(row.id+".*"))
        image=read_image(image_path);assert image is not None
        h,w=image.shape[:2]; gt=[]
        for line in (DATASET/split/"labels"/(row.id+".txt")).read_text().splitlines():
            _,x,y,bw,bh=map(float,line.split())
            gt.append(((x-bw/2)*w,(y-bh/2)*h,(x+bw/2)*w,(y+bh/2)*h))
        start=time.perf_counter()
        out=model.predict(image,conf=.01,iou=.7,device=device,verbose=False)[0]
        ms=1000*(time.perf_counter()-start)
        boxes=out.boxes.xyxy.cpu().numpy();scores=out.boxes.conf.cpu().numpy()
        rows.append((row.id,gt,boxes,scores,ms))
    return rows
def measure(rows,threshold):
    tp=fp=fn=0;errors=[];exact=[];positive=[];ms=[]
    for _,gt,boxes,scores,elapsed in rows:
        predicted=boxes[scores>=threshold].tolist();a,b,c=match(gt,predicted)
        tp+=a;fp+=b;fn+=c;errors.append(abs(len(gt)-len(predicted)))
        exact.append(len(gt)==len(predicted));ms.append(elapsed)
        if gt:positive.append(len(gt)==len(predicted))
    precision=tp/(tp+fp) if tp+fp else 0.; recall=tp/(tp+fn) if tp+fn else 0.
    return {"limiar":threshold,"imagens":len(rows),"positivas":len(positive),"tp":tp,"fp":fp,"fn":fn,
            "precisao":precision,"recall":recall,"f1":2*precision*recall/(precision+recall) if precision+recall else 0.,
            "contagem_exata":float(np.mean(exact)),"contagem_exata_positivas":float(np.mean(positive)) if positive else None,
            "erro_absoluto_medio":float(np.mean(errors)),"tempo_medio_ms":float(np.mean(ms))}
assert iou([0,0,10,10],[0,0,10,10])==1
assert match([[0,0,10,10]],[[0,0,10,10],[0,0,10,10]])==(1,1,0)


## Seleção na validação

In [ ]:
val=infer_split("val")
thresholds=[.01,.025,.05,.075,.10,.15,.20,.25,.35,.50,.65]
validation=pd.DataFrame([measure(val,t) for t in thresholds])
best=validation.sort_values(["f1","contagem_exata_positivas"],ascending=False).iloc[0]
threshold=float(best.limiar)
validation.to_csv(OUTPUT/"limiares_validacao.csv",index=False)
(OUTPUT/"limiar_escolhido_validacao.json").write_text(json.dumps({"limiar":threshold},indent=2),encoding="utf8")
print("Limiar escolhido na validação:",threshold)
display(validation)


## Teste reservado

In [ ]:
test=infer_split("test")
final=measure(test,threshold)
(OUTPUT/"teste_limiar_calibrado.json").write_text(json.dumps(final,indent=2,ensure_ascii=False),encoding="utf8")
display(pd.DataFrame([final]))


O tempo inclui inferência e supressão com confiança mínima 0,01, mas não a leitura de faixas. Resultados nesta divisão não demonstram independência de cena e componente físico: veja as limitações no notebook 1.